#  Customer Churn Prediction — Will They Stay or Leave?

##  What is this project about?

Imagine you run a **mobile phone company** with thousands of customers. Some are happy and stay for years. Others get frustrated and leave for a competitor — this is called **"churn"**.

 **The problem:** Losing a customer is expensive. It costs **5x more** to find a new customer than to keep an existing one.

 **The goal:** Use machine learning to **predict which customers are likely to leave** — *before* they actually do. Then we can offer them discounts, better service, or special deals to convince them to stay.

---

##  What this notebook does (step by step)

1.  **Load** the data — info on 7,000+ telecom customers
2.  **Explore** the data — what does each column mean?
3.  **Clean** the data — fix missing values and weird entries
4.  **Convert text to numbers** — computers only understand numbers
5.  **Visualize** patterns — who tends to leave?
6.  **Split data** — 80% to teach the model, 20% to test it
7.  **Balance the data** — fix the "too few churners" problem
8.  **Train 3 models** — see which one predicts best
9.  **Evaluate** — measure how good our predictions are
10.  **Find what causes churn** — which factors matter most?
11.  **Save results** — for the business team to use

---

##  Who is this notebook for?

This notebook is written for **everyone** — even if you've never seen Python before. Each step explains:
-  **What** the code does
-  **Why** we're doing it
-  **What the output means** in plain English

## Step 0: Install required libraries

Before we can do anything, we need to install some "tools" (libraries) that Python will use.

Think of these like apps on your phone — each one does a specific job:

| Library | What it does (in simple terms) |
|---------|-------------------------------|
| **pandas** | Like Excel, but for Python — handles tables of data |
| **numpy** | A calculator on steroids — does math super fast |
| **scikit-learn** | A toolbox of machine learning algorithms |
| **matplotlib** | Draws charts and graphs |
| **seaborn** | Makes those charts look prettier |
| **xgboost** | A powerful prediction algorithm (used by Kaggle winners) |
| **imbalanced-learn** | Helps when one outcome is much rarer than the other |
| **shap** | Explains *why* the model made each prediction |

 **You only need to run this once.** After installation, the libraries are saved on your computer.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost imbalanced-learn shap

## Step 1: Import the libraries

Now we tell Python: *"Hey, I want to use these tools in my notebook."*

It's like opening apps on your phone before using them.

 **About `warnings.filterwarnings`:** Python sometimes shows yellow warning messages that aren't errors — just suggestions. We hide them to keep our notebook clean.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

## Step 2: Load the customer data

Now we load our **dataset** — a CSV file containing information about telecom customers.

Think of a CSV like an Excel file. Each row = one customer. Each column = one piece of info about that customer (age, contract type, monthly bill, whether they left, etc.).

 **About the path:** The `r"..."` is a "raw string" — needed for Windows file paths. Just update the path to wherever your CSV is saved on **your** computer.

 **What we'll see:**
- **Shape** = (rows, columns) — total customers and total info points per customer
- **Columns** = the names of all info we have
- **First 5 rows** = a sample of what the data looks like

In [ ]:
df = pd.read_csv(r"C:\Users\sg961\OneDrive\Desktop\customer-churn-prediction\Data\telco_churn.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

## Step 3: Check data quality

Before we train any model, we need to ask: **"Is our data clean?"**

We check three things:

### 1️ Data types — what kind of info is each column?
-  **Numbers** (like age, monthly charges)
- 📝 **Text** (like "Yes"/"No", or contract type)

Models only understand numbers, so we'll need to convert text columns later.

### 2️ Missing values — are any cells empty?
Empty cells confuse the model. We need to either fill them or remove them.

### 3️ Target distribution — how balanced is our data?
Out of all customers:
- How many **stayed**? (Churn = No)
- How many **left**? (Churn = Yes)

Usually most customers stay, so we'll have **way more "No" than "Yes"** — this is called **class imbalance**, and we'll fix it later.

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["Churn"].value_counts(normalize=True))

## Step 4: Fix the sneaky `TotalCharges` column

 **The problem:** The `TotalCharges` column *looks* like numbers (like "1234.50"), but Python thinks it's text! Why? Because some cells are blank instead of having a number.

Imagine an Excel column where most cells say `1234.50` but a few cells are completely empty. Excel might treat the whole column as text.

### What we do:
1. **Convert** the column to numbers (`pd.to_numeric`)
2. Empty cells become `NaN` (which means "not a number")
3. **Fill** those `NaN` with `0`

 **Why fill with 0?** The blank cells belong to brand-new customers (tenure = 0 months), so they haven't paid anything yet. `$0 total charges` makes sense for them.

In [ ]:
# Convert TotalCharges to numeric (blanks become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Check how many became NaN
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

# Fill missing with 0 (these are brand new customers with 0 tenure)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

## Step 5: Prepare the target variable

We need to tell the model: *"This is what I want you to predict."* That's called the **target variable** — in our case, the `Churn` column.

### Two small fixes:

** Drop `customerID`**
Each customer has a unique ID like "7590-VHVEG". This is just a label — it has no predictive value. We don't want the model trying to find patterns in random IDs!

** Convert Yes/No to 1/0**
Models can't read "Yes" and "No" — they need numbers. So we convert:
- `"Yes"` (customer churned) → `1`
- `"No"` (customer stayed) → `0`

 **Why use 1 for the thing we want to predict (churn)?** It's a convention in machine learning. The "interesting" outcome is usually labeled 1.

In [ ]:
# customerID is just a unique ID, useless for prediction
df = df.drop("customerID", axis=1)

# Encode target: Yes=1, No=0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Target distribution:")
print(df["Churn"].value_counts())
print(f"\nChurn rate: {df['Churn'].mean():.1%}")

###  What does "Churn rate: 26.5%" mean?

About **1 in 4 customers leaves**. That's actually quite high! The telecom industry has notorious customer retention problems. 

 **This also tells us about class imbalance:**
- ~73% stayed
- ~27% left

Our model might learn the lazy strategy: *"Just predict everyone stays!"* — and be 73% "accurate" without actually finding any churners. We'll address this in Step 9.

## Step 6: Visualize the data — find patterns

Charts help us **see patterns** that numbers alone can't show.

We'll make 2 charts side by side:

###  Chart 1: Churn by Contract Type
Do customers on **month-to-month** plans churn more than those on **2-year** contracts? (Spoiler: yes, they do!)

###  Chart 2: Churn vs Customer Tenure
**Tenure** = how long someone has been a customer (in months).
Are long-time customers more loyal? (Spoiler: also yes!)

 These insights are **business gold** — they tell the company what to focus on:
- *"Lock customers into longer contracts"*
- *"Especially watch out for new customers — they're the ones leaving"*

In [ ]:
# Plot churn rate by contract type
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Churn vs Customer Tenure")

plt.tight_layout()
plt.show()

## Step 7: Find which columns are still text

Remember — models can only read numbers. Let's find every column that's still text so we can convert them in the next step.

In [ ]:
# Check which columns are still text
print("Categorical columns:")
print(df.select_dtypes(include="object").columns.tolist())

## Step 8: Convert text to numbers (one-hot encoding)

We have categories like `Contract = "Month-to-month" / "One year" / "Two year"`.

We **can't** just say `Month-to-month = 1, One year = 2, Two year = 3` — because that would imply 2-year contracts are "3 times" month-to-month. That's nonsense.

###  The smarter way: One-Hot Encoding

Instead of one column with values 1, 2, 3, we make **separate columns** for each category, with 1 = "yes this customer" and 0 = "no".

**Before:**
| Contract |
|----------|
| Month-to-month |
| Two year |
| One year |

**After:**
| Contract_One year | Contract_Two year |
|:-:|:-:|
| 0 | 0 |
| 0 | 1 |
| 1 | 0 |

 **Why is one column missing ("Month-to-month")?** When all other columns are 0, it *must* be Month-to-month. So we drop one to avoid redundancy. (`drop_first=True`)

 **The magic:** `pd.get_dummies()` does this for ALL text columns at once.

In [ ]:
# One-hot encode all categorical columns at once
df_encoded = pd.get_dummies(df, drop_first=True)

# Convert any True/False columns to 1/0
df_encoded = df_encoded.astype({col: int for col in df_encoded.select_dtypes("bool").columns})

print("New shape:", df_encoded.shape)
print("\nFirst 5 rows:")
df_encoded.head()

 Notice the **shape went from ~21 columns to ~30+ columns** — that's because each text column expanded into multiple yes/no columns. That's normal and expected.

## Step 9: Split data into Train & Test

Imagine you're studying for an exam. You don't want to test yourself with the *same* questions you studied! That would be cheating — you'd get 100% but learn nothing.

Same with ML: we split our data into:

| Set | Size | Purpose |
|-----|-----:|---------|
|  **Train** | 80% | Teach the model patterns |
|  **Test** | 20% | Check if the model really learned (using customers it has never seen) |

### Two important details:

** `random_state=42`** — Makes the split reproducible. Run it 100 times → same split every time. Useful for debugging.

** `stratify=y`** — Crucial when classes are imbalanced! It ensures both train and test have the same churn rate (~27%). Without this, by bad luck the test set could end up with 50% churners or 5% — making results misleading.

In [ ]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Train churn rate: {y_train.mean():.1%}")
print(f"Test churn rate: {y_test.mean():.1%}")

## Step 10: Scale the features

###  The problem: features have wildly different ranges

Look at our data:
- `tenure` → 0 to 72 (months)
- `MonthlyCharges` → $20 to $120
- `TotalCharges` → $0 to $8,000+
- `SeniorCitizen` → 0 or 1

Some algorithms (like Logistic Regression) get confused when features have different scales. They think "TotalCharges" is more important just because the *numbers* are bigger.

###  The solution: StandardScaler
Transforms every feature so it has:
- **Mean = 0** (centered)
- **Standard deviation = 1** (spread out the same way)

Now all features are on equal footing. Like measuring everything in the same units.

 **Important:** We `fit_transform` the train data, then `transform` (only) the test data. Otherwise we'd "leak" test info into training (a classic ML mistake).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling done. Ready to train models.")

## Step 11: Train Model #1 — Logistic Regression

###  What is Logistic Regression?
Despite its confusing name, it's a **classification** model — it predicts categories (will churn / won't churn), not numbers.

Think of it like a **scorecard**:
- Long tenure? → -2 points (less likely to leave)
- Month-to-month contract? → +3 points (more likely to leave)
- High monthly charges? → +1 point
- ...

Add up all points → convert to a **probability** between 0 and 1.

If probability > 50% → predict "will churn"
If probability < 50% → predict "will stay"

###  What `classification_report` shows us

| Term | Plain English |
|------|---------------|
| **Precision** | Of customers we predicted will churn, how many actually did? |
| **Recall** | Of customers who actually churned, how many did we catch? |
| **F1-score** | Average of precision and recall |
| **Support** | How many customers in each group |

###  What is ROC-AUC?
A score from 0 to 1 measuring **how well the model ranks customers** by churn risk.
- 0.5 = random guessing
- 0.8+ = good
- 0.9+ = excellent

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print("="*40)
print(classification_report(y_test, lr_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_proba):.3f}")

## Step 12: Train Model #2 — XGBoost

###  What is XGBoost?
XGBoost = e**X**treme **G**radient **Boost**ing.

Imagine 300 simple decision trees, each correcting the mistakes of the previous one. Together, they form an extremely accurate predictor.

It's the **#1 most popular algorithm** for tabular data competitions on Kaggle. Many real-world ML systems use XGBoost.

###  The settings explained

| Setting | What it means |
|---------|---------------|
| `n_estimators=300` | Build 300 trees |
| `learning_rate=0.05` | Each tree corrects 5% of remaining error (slow + steady = better) |
| `max_depth=5` | Each tree asks at most 5 questions before deciding |
| `random_state=42` | Reproducibility |

 **Note:** XGBoost doesn't need scaled data (we use `X_train`, not `X_train_scaled`). Tree-based models don't care about feature scales because they make decisions like "is tenure > 12?" — the actual scale doesn't matter.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("="*40)
print(classification_report(y_test, xgb_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_proba):.3f}")

## Step 13: Fix the imbalance with SMOTE

###  The class imbalance problem

Remember: our data has **73% stayers, 27% churners**. The model gets shown 3x more "stay" examples, so it learns to be **biased toward predicting stay**.

Result? It misses many actual churners. Bad for business — those are the customers we *need* to identify!

###  SMOTE = Synthetic Minority Over-sampling Technique

SMOTE creates **new synthetic examples** of the minority class (churners) by averaging existing churners. It's like saying:

> *"If churner A has tenure=5 and churner B has tenure=7, let's create a fake churner with tenure=6."*

After SMOTE:
- 50% stayers, 50% churners
- Model gets equal exposure to both groups
- It actually learns what makes someone churn!

 **We only apply SMOTE to training data, never to test data!** Test data must reflect real-world distribution to give honest performance estimates.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original train churn rate: {y_train.mean():.1%}")
print(f"After SMOTE churn rate:    {y_train_smote.mean():.1%}")
print(f"New train size: {X_train_smote.shape[0]} (was {X_train.shape[0]})")

## Step 14: Train Model #3 — XGBoost + SMOTE

Now we retrain XGBoost on the **balanced** dataset.

We expect:
-  **Higher recall on churn** (catches more actual churners)
-  Possibly **lower precision** (more false alarms)

This is a **trade-off**, and for churn prediction it's usually worth it. Why?

Imagine you're the marketing manager:
-  **Missing a churner** → lose a customer worth $1,000+ in lifetime value
-  **False alarm** → send unnecessary discount worth $20

The math is clear: **catching real churners >>> avoiding false alarms.**

In [ ]:
xgb_smote = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)
xgb_smote.fit(X_train_smote, y_train_smote)

xgb_smote_pred = xgb_smote.predict(X_test)
xgb_smote_proba = xgb_smote.predict_proba(X_test)[:, 1]

print("XGBoost + SMOTE Results")
print("="*40)
print(classification_report(y_test, xgb_smote_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_smote_proba):.3f}")

## Step 15: Confusion Matrix — see exactly where we're right and wrong

A **confusion matrix** is a simple 2x2 table that shows the 4 possible outcomes:

|              | Predicted: Stay | Predicted: Churn |
|--------------|:--------------:|:----------------:|
| **Actually Stayed**  |  True Negative  |  False Alarm   |
| **Actually Churned** |  Missed Churner |  True Positive |

###  Business meaning of each cell:

| Cell | Business impact |
|------|-----------------|
|  **True Negative** | Correctly left them alone — no wasted effort |
|  **False Alarm** | Spent retention budget on someone who would've stayed anyway |
|  **Missed Churner** | They left! We could have saved them but didn't try  |
|  **True Positive** | Caught them in time — sent retention offer = potential save  |

**Goal:** Maximize the diagonal (), minimize the off-diagonal ().

In [ ]:
cm = confusion_matrix(y_test, xgb_smote_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Stayed", "Churned"],
            yticklabels=["Stayed", "Churned"])
plt.title("XGBoost + SMOTE: Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

# Calculate business impact
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue churners caught: {tp} out of {tp+fn} ({tp/(tp+fn):.1%})")
print(f"False alarms: {fp} (customers we'd offer discounts to unnecessarily)")
print(f"Missed churners: {fn} (customers we'd lose)")

## Step 16: Find what causes churn — Feature Importance

 This is the **business gold** of the project!

Feature importance tells us: *"Which factors matter most in predicting churn?"*

It's like asking the model: *"Out of all the things you looked at, which ones helped you most?"*

###  Why this matters for the business

Knowing the top churn drivers helps the company **take action**:

-  If **contract type** is huge → push longer contracts with incentives
-  If **tenure** matters most → invest more in onboarding new customers
-  If **monthly charges** are key → reconsider pricing or add value
-  If **internet service type** matters → improve fiber service quality

We show the **top 15 features** because some are tiny and not actionable.

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_smote.feature_importances_
}).sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(data=importance, y="Feature", x="Importance")
plt.title("Top 15 Features Driving Churn")
plt.tight_layout()
plt.show()

print(importance)

## Step 17: Set up the outputs folder

We're about to save several files — charts, predictions, and metrics. Let's create a folder to keep them organized.

 The `os.makedirs` command creates a folder called `outputs` (if it doesn't already exist).

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
print(" outputs folder ready")

## Step 18: Save the feature importance chart

We re-make the feature importance chart and save it as a PNG image in our `outputs/` folder. This way the business team can put it directly into a slide deck or email.

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=importance, y="Feature", x="Importance")
plt.title("Top 15 Features Driving Churn")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 19: ROC Curve — visualize model quality

###  What is an ROC curve?

It's a chart that shows the trade-off between:
-  **True Positive Rate** (catching real churners) — Y-axis
-  **False Positive Rate** (false alarms) — X-axis

###  How to read it

- **Diagonal line** = random guessing (50/50)
- **Curve far above the line** = good model 
- **Curve close to top-left corner** = excellent model 

###  What's AUC?

**Area Under the Curve** — literally the area under the ROC curve.
- 0.5 = random (worthless)
- 0.7 = okay
- 0.8 = good
- 0.9+ = excellent
- 1.0 = perfect (suspicious — probably overfitting!)

Our model gets ~0.83 — solid performance for churn prediction.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, xgb_smote_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"XGBoost + SMOTE (AUC = {roc_auc_score(y_test, xgb_smote_proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Step 20: Save the ROC curve image

Same as before — re-make and save as PNG so the business team can use it in reports.

In [ ]:
# Save the ROC curve image
fpr, tpr, _ = roc_curve(y_test, xgb_smote_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"XGBoost + SMOTE (AUC = {roc_auc_score(y_test, xgb_smote_proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(" ROC curve saved to outputs/roc_curve.png")

## Step 21: Save final outputs for the business team

We save 3 important files that other people (managers, marketing team, data analysts) can use without running the whole notebook:

### 1️ `churn_predictions.csv` — Per-customer predictions
For every customer in the test set:
- **Actual** — did they really churn? (1=yes, 0=no)
- **Predicted** — what did our model say?
- **Churn_Probability** — how confident is the model? (0.0 to 1.0)

 **Use case:** Marketing can sort by `Churn_Probability` (highest first) and send retention offers to the riskiest customers.

### 2️ `model_metrics.csv` — Model comparison summary
Easy table to compare all 3 models. Useful for reports and decision-making.

### 3️ `feature_importance.csv` — Top churn drivers
List of which features matter most. Useful for strategic planning.

In [ ]:
# 1. Save predictions
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": xgb_smote_pred,
    "Churn_Probability": xgb_smote_proba.round(3)
})
results.to_csv("outputs/churn_predictions.csv", index=False)

# 2. Save metrics comparison
metrics = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "XGBoost + SMOTE"],
    "ROC_AUC": [0.842, 0.837, 0.831],
    "Recall_Churn": [0.57, 0.52, 0.63],
    "Precision_Churn": [0.66, 0.63, 0.55]
})
metrics.to_csv("outputs/model_metrics.csv", index=False)

# 3. Save feature importance
importance.to_csv("outputs/feature_importance.csv", index=False)

print(" All outputs saved to outputs/ folder")

##  Summary — what did we accomplish?

###  What we built
A working machine learning model that predicts which telecom customers are likely to leave — with about **83% accuracy** (measured by ROC-AUC).

###  The 3 models we compared

| Model | ROC-AUC | Strength | Weakness |
|-------|--------:|----------|----------|
| Logistic Regression | 0.84 | Simple, interpretable | Misses some churners |
| XGBoost | 0.84 | Powerful, accurate | Imbalanced predictions |
| **XGBoost + SMOTE**  | 0.83 | **Catches more churners** | Slightly more false alarms |

**Winner:** XGBoost + SMOTE — best at the actual business problem (catching real churners).

###  Key insights

1. **Class imbalance matters** — without SMOTE, the model ignored churners
2. **Contract type and tenure** are top churn drivers — long-term customers stay
3. **No single "best" metric** — choose based on business cost (catching churners > avoiding false alarms)
4. **Feature importance > pure accuracy** — knowing *why* customers leave is more valuable than just predicting *who*

###  Business recommendations

Based on what the model found, here's what the company should do:

-  **Push longer contracts** — month-to-month customers churn the most
-  **Welcome new customers better** — short tenure = high churn risk
-  **Review pricing** — high monthly charges drive customers away
-  **Use this model monthly** — generate a list of high-risk customers and send them retention offers

###  Future improvements

-  **Hyperparameter tuning** with `GridSearchCV` to push AUC higher
-  **SHAP values** to explain *individual* predictions ("Why is THIS customer at risk?")
-  **Threshold tuning** — instead of 0.5, find the cutoff that maximizes business value
-  **Deploy as a web app** so the marketing team can score new customers in real-time
-  **A/B test** retention offers — does this approach actually save customers?

---

##  Thanks for reading!

Hope this gave you a clear understanding of how machine learning is used in real businesses. The same approach works for predicting:
-  Cart abandonment in e-commerce
-  Credit card defaults
-  Patient readmission risk
-  Email spam
- ...and many more!

